In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import boto3
import pickle
from api import map_ltv_range_to_lgd_bin

try:
    import optbinning
except:
    ! pip install optbinning
    
try:
    import catboost
except:
    ! pip install catboost

try:
    import xmltodict
except:
    ! pip install xmltodict

#### Functions

In [2]:
def get_lgd_bk_nobk(int_bk, flt_lgd_bk, flt_lgd_nobk):
    # if bk
    if int_bk == 1:
        return flt_lgd_bk
    else:
        return flt_lgd_nobk

#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

Project: 20250221-credit-builder-analysis
Task: 02_gen13_predictions


#### Output dir

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,1,0,7,NaN,1.311220,1,0,0,0,0.616667
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1,0,5,NaN,1.432368,0,0,0,0,NaN
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,1,0,4,NaN,1.587073,1,0,0,0,NaN
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,0,2,NaN,1.081881,0,1,0,1,0.000000
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1,0,4,NaN,1.371350,0,1,0,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,4,NaN,1.280957,0,0,0,0,0.000000
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1,0,0,NaN,1.198869,1,0,0,0,0.367073
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,1,0,3,NaN,1.313231,0,0,0,0,NaN
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,1,0,3,NaN,0.991586,1,0,0,0,0.006585


#### Import parser

In [6]:
str_filename = 'cls_parser.pkl'
str_local_path = f'./{str_filename}'
cls_parser = pickle.load(open(str_local_path, 'rb'))

#### Preprocess data

In [7]:
cls_model_preprocessing = cls_parser.cls_model_preprocessing
df = cls_model_preprocessing.transform(df)
# show
df

Masking negative values to NaN...


0it [00:00, ?it/s]


Capping income...
Replacing zeros...


100%|██████████| 3/3 [00:00<00:00, 757.73it/s]


Engineering number of months...
Engineering number of months total...
Engineering weighted average...
Engineering tag for has auto...
Engineering tag for open auto indicator...
Engineering tag for closed auto indicator...
Engineering tag for open and closed auto indicator...
Engineering 3 month early delinquency...
Engineering 6 month early delinquency...
Engineering 3 month recent delinquency...
Engineering 6 month recent delinquency...
Engineering DTI...
Engineering franchise...
Engineering has a codebtor...
Engineering vehicle age...
Engineering PTI...
Engineering LTV...
Engineering BK...
Engineering perfect payment history tag for most recent auto...
Engineering perfect payment history tag for open auto...
Engineering perfect payment history tag for closed auto...
Engineering interactions...
Imputing values...


100%|██████████| 1123/1123 [00:01<00:00, 622.85it/s]


Binning values for scorecard...


100%|██████████| 1119/1119 [00:05<00:00, 187.39it/s]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,addrchangecount06month__ln_binned,addrchangecount12month__ln_binned,addrchangecount24month__ln_binned,addrchangecount60month__ln_binned,addrlastmovetaxratiodiff__ln_binned,addrlastmoveecontrajectory__ln_binned,addrlastmoveecontrajectoryindex__ln_binned,phoneinputproblems__ln_binned,phoneinputsubjectcount__ln_binned,alertregulatorycondition__ln_binned
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.0,-0.211617,-0.158847,-0.289687,0.0,-0.097823,-0.134102,-0.039639,-0.038041,0.216812
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.0,0.048333,0.108514,0.032394,0.0,-0.097823,-0.091170,0.032249,0.028913,0.216812
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.048333,0.108514,0.225637,0.0,0.211757,0.172319,-0.039639,-0.038041,0.216812
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0.0,0.048333,-0.158847,-0.164347,0.0,-0.097823,-0.091170,0.032249,0.028913,0.216812
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0.0,0.048333,0.108514,0.225637,0.0,0.211757,0.172319,0.032249,0.028913,-0.107256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0.0,0.048333,0.108514,0.032394,0.0,-0.097823,0.172319,0.032249,0.028913,-0.107256
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,0.0,0.048333,0.108514,0.032394,0.0,-0.097823,-0.091170,0.032249,0.028913,0.216812
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,0.0,0.048333,0.108514,0.032394,0.0,-0.097823,-0.134102,-0.039639,-0.038041,0.216812
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,0.0,0.048333,0.108514,0.032394,0.0,-0.097823,-0.134102,0.032249,0.028913,0.216812


#### PD predictions

In [8]:
cls_model_inference = cls_parser.cls_model_inference
# get intercept
flt_intercept = cls_model_inference.intercept_[0]
# get cols in model
list_cols_model = list(cls_model_inference.feature_names_in_)
# get the coef
list_coef = list(cls_model_inference.coef_[0])
# make dict
dict_coef = dict(zip(list_cols_model, list_coef))
# get contribution
list_str_contribution = []
for key, val in dict_coef.items():
    str_contribution = f'{key}_contribution'
    df[str_contribution] = df[key] * val 
    list_str_contribution.append(str_contribution)
# get the sum
df['sum'] = df[list_str_contribution].apply(sum, axis=1)
# get the log odds
df['log_odds'] = df['sum'] + flt_intercept
# get the pd
df['pd'] = np.exp(df['log_odds']) / (1 + np.exp(df['log_odds']))
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,linka006__tu_binned_contribution,inquiryshortterm12month__ln_binned_contribution,addrinputsubjectcount__ln_binned_contribution,g251c__tu_binned_contribution,g095s__tu_binned_contribution,s209a__tu_binned_contribution,inquirynonshortterm12month__ln_binned_contribution,sum,log_odds,pd
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.034744,-0.040978,0.014632,0.022693,-0.142548,0.018359,0.033394,0.378121,0.317318,0.578671
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.034744,-0.040978,-0.014258,-0.128272,-0.142548,-0.037731,-0.061136,-0.541494,-0.602297,0.353818
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,-0.065002,-0.040978,-0.014258,-0.128272,0.059176,0.018359,0.033394,-0.182301,-0.243103,0.439522
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0.034744,0.147799,-0.014258,0.022693,-0.142548,0.018359,0.033394,-0.141410,-0.202212,0.449619
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0.034744,-0.040978,0.014632,-0.128272,0.059176,0.018359,-0.061136,-0.556404,-0.617206,0.350417
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0.034744,-0.040978,-0.014258,0.022693,0.059176,0.018359,-0.061136,0.969894,0.909092,0.712814
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,0.034744,-0.040978,0.004595,0.022693,-0.142548,0.018359,0.033394,-0.421182,-0.481984,0.381784
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,-0.018353,0.147799,0.004595,0.022693,-0.142548,0.018359,0.033394,0.582955,0.522153,0.627651
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,-0.030403,-0.040978,0.014632,0.022693,-0.142548,-0.120138,0.033394,-0.528978,-0.589780,0.356685


#### LGD

In [9]:
# bk
dict_bins_ltv = cls_parser.dict_bins_ltv_bk
df['lgd_bk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv,
    ),
)
# show
#df

In [10]:
# non bk
dict_bins_ltv = cls_parser.dict_bins_ltv_nobk
df['lgd_nobk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv,
    ),
)
# show
#df

In [11]:
# choose appropriate ltv
df['lgd'] = df.apply(
    lambda x: get_lgd_bk_nobk(
        int_bk=x['ENG-bk'],
        flt_lgd_bk=x['lgd_bk'],
        flt_lgd_nobk=x['lgd_nobk'],
    ),
    axis=1,
)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,g251c__tu_binned_contribution,g095s__tu_binned_contribution,s209a__tu_binned_contribution,inquirynonshortterm12month__ln_binned_contribution,sum,log_odds,pd,lgd_bk,lgd_nobk,lgd
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.022693,-0.142548,0.018359,0.033394,0.378121,0.317318,0.578671,0.323245,0.390043,0.323245
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,-0.128272,-0.142548,-0.037731,-0.061136,-0.541494,-0.602297,0.353818,0.355509,0.390043,0.390043
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,-0.128272,0.059176,0.018359,0.033394,-0.182301,-0.243103,0.439522,0.416230,0.416881,0.416230
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0.022693,-0.142548,0.018359,0.033394,-0.141410,-0.202212,0.449619,0.259641,0.337281,0.337281
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,-0.128272,0.059176,0.018359,-0.061136,-0.556404,-0.617206,0.350417,0.333463,0.390043,0.390043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0.022693,0.059176,0.018359,-0.061136,0.969894,0.909092,0.712814,0.316725,0.390043,0.390043
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,0.022693,-0.142548,0.018359,0.033394,-0.421182,-0.481984,0.381784,0.289997,0.350185,0.289997
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,0.022693,-0.142548,0.018359,0.033394,0.582955,0.522153,0.627651,0.323245,0.390043,0.390043
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,0.022693,-0.142548,-0.120138,0.033394,-0.528978,-0.589780,0.356685,0.172898,0.277756,0.172898


#### Convert non-numeric to string

In [12]:
for col in tqdm(df.columns):
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 3940/3940 [00:18<00:00, 211.42it/s] 


#### Write to s3

In [13]:
%%time

str_filename = 'df_clean_w_pred.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)

CPU times: user 41.7 s, sys: 509 ms, total: 42.2 s
Wall time: 47.4 s
